# System 2105

In [37]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 20.79253,
	"longitude": -156.512,
	"start_date": "2018-12-15",
	"end_date": "2023-11-15",
	"daily": "sunshine_duration",
	"hourly": "direct_radiation",
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_direct_radiation = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["direct_radiation"] = hourly_direct_radiation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunshine_duration = daily.Variables(0).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

Coordinates: 20.843584060668945°N -156.49798583984375°E
Elevation: 14.0 m asl
Timezone: b'Pacific/Honolulu'b'GMT-10'
Timezone difference to GMT+0: -36000s

Hourly data
                            date  direct_radiation
0     2018-12-15 00:00:00+00:00               0.0
1     2018-12-15 01:00:00+00:00               0.0
2     2018-12-15 02:00:00+00:00               0.0
3     2018-12-15 03:00:00+00:00               0.0
4     2018-12-15 04:00:00+00:00               0.0
...                         ...               ...
43123 2023-11-15 19:00:00+00:00               0.0
43124 2023-11-15 20:00:00+00:00               0.0
43125 2023-11-15 21:00:00+00:00               0.0
43126 2023-11-15 22:00:00+00:00               0.0
43127 2023-11-15 23:00:00+00:00               0.0

[43128 rows x 2 columns]

Daily data
                           date  sunshine_duration
0    2018-12-15 00:00:00+00:00       38617.980469
1    2018-12-16 00:00:00+00:00       38643.386719
2    2018-12-17 00:00:00+00:00       38669

In [38]:
hourly_dataframe.describe()

,direct_radiation
count,43128.000000
mean,164.131561
std,231.324783
min,0.000000
25%,0.000000
50%,4.000000
75%,314.000000
max,978.000000


In [39]:
#there's no radiation at night (duh)
#may want to take daily averages of radiation
hourly_dataframe

,date,direct_radiation
0,2018-12-15 00:00:00+00:00,0.0
1,2018-12-15 01:00:00+00:00,0.0
2,2018-12-15 02:00:00+00:00,0.0
3,2018-12-15 03:00:00+00:00,0.0
4,2018-12-15 04:00:00+00:00,0.0
...,...,...
43123,2023-11-15 19:00:00+00:00,0.0
43124,2023-11-15 20:00:00+00:00,0.0
43125,2023-11-15 21:00:00+00:00,0.0
43126,2023-11-15 22:00:00+00:00,0.0


In [40]:
hourly_dataframe["day"] = hourly_dataframe['date'].dt.date
hourly_dataframe = hourly_dataframe.loc[hourly_dataframe['direct_radiation']>0]
hourly_dataframe

,date,direct_radiation,day
8,2018-12-15 08:00:00+00:00,51.0,2018-12-15
9,2018-12-15 09:00:00+00:00,225.0,2018-12-15
10,2018-12-15 10:00:00+00:00,402.0,2018-12-15
11,2018-12-15 11:00:00+00:00,554.0,2018-12-15
12,2018-12-15 12:00:00+00:00,666.0,2018-12-15
...,...,...,...
43118,2023-11-15 14:00:00+00:00,567.0,2023-11-15
43119,2023-11-15 15:00:00+00:00,432.0,2023-11-15
43120,2023-11-15 16:00:00+00:00,311.0,2023-11-15
43121,2023-11-15 17:00:00+00:00,104.0,2023-11-15


In [41]:
daily_mean_direct_radiation = hourly_dataframe.groupby("day")['direct_radiation'].mean()


In [42]:
df = pd.DataFrame(daily_mean_direct_radiation)
df = df.reset_index()
df

,day,direct_radiation
0,2018-12-15,388.818176
1,2018-12-16,384.818176
2,2018-12-17,278.818176
3,2018-12-18,318.545441
4,2018-12-19,280.636353
...,...,...
1792,2023-11-11,248.500000
1793,2023-11-12,358.166656
1794,2023-11-13,310.333344
1795,2023-11-14,191.416672


In [43]:
#check for skipped days
df['date_diff'] = df['day'].diff()
df[df["date_diff"] > pd.Timedelta(days=1)]

,day,direct_radiation,date_diff


In [44]:
df_daily = daily_dataframe.rename(columns = {'date':'day'})
df_daily['day'] = df_daily['day'].dt.date
df_daily['direct_radiation'] = df['direct_radiation']
df_daily

,day,sunshine_duration,direct_radiation
0,2018-12-15,38617.980469,388.818176
1,2018-12-16,38643.386719,384.818176
2,2018-12-17,38669.789062,278.818176
3,2018-12-18,38464.207031,318.545441
4,2018-12-19,38659.699219,280.636353
...,...,...,...
1792,2023-11-11,38536.957031,248.500000
1793,2023-11-12,38643.359375,358.166656
1794,2023-11-13,38499.261719,310.333344
1795,2023-11-14,37654.582031,191.416672


In [45]:
df_daily  = daily_dataframe.rename(columns = {'date':'time'})
df_daily['time'] = df_daily['time'].dt.date
df_daily
df_daily.to_csv("prize_weather/2105.csv", index = False)


## 2107

In [36]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 38.996306,
	"longitude": -122.134111,
	"start_date": "2017-01-01",
	"end_date": "2024-11-30",
	"daily": "sunshine_duration",
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunshine_duration = daily.Variables(0).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

df_daily  = daily_dataframe.rename(columns = {'date':'time'})
df_daily['time'] = df_daily['time'].dt.date
df_daily
df_daily.to_csv("prize_weather/2107.csv", index = False)

Coordinates: 38.98066711425781°N -122.17808532714844°E
Elevation: 110.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s

Daily data
                           date  sunshine_duration
0    2017-01-01 00:00:00+00:00       13963.484375
1    2017-01-02 00:00:00+00:00       32922.507812
2    2017-01-03 00:00:00+00:00           0.000000
3    2017-01-04 00:00:00+00:00       14935.264648
4    2017-01-05 00:00:00+00:00       33077.699219
...                        ...                ...
2886 2024-11-26 00:00:00+00:00       34316.789062
2887 2024-11-27 00:00:00+00:00       34367.921875
2888 2024-11-28 00:00:00+00:00       34289.539062
2889 2024-11-29 00:00:00+00:00       34213.671875
2890 2024-11-30 00:00:00+00:00       34140.343750

[2891 rows x 2 columns]


# System 7333

In [35]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 36.578000,
	"longitude": -120.401600,
	"start_date": "2016-10-31",
	"end_date": "2023-10-31",
	"daily": "sunshine_duration",
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunshine_duration = daily.Variables(0).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

df_daily  = daily_dataframe.rename(columns = {'date':'time'})
df_daily['time'] = df_daily['time'].dt.date
df_daily
df_daily.to_csv("prize_weather/7333.csv", index = False)

Coordinates: 36.59050750732422°N -120.39266967773438°E
Elevation: 69.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s

Daily data
                           date  sunshine_duration
0    2016-10-31 00:00:00+00:00       37194.937500
1    2016-11-01 00:00:00+00:00       36003.304688
2    2016-11-02 00:00:00+00:00       36887.281250
3    2016-11-03 00:00:00+00:00       36861.148438
4    2016-11-04 00:00:00+00:00       36224.851562
...                        ...                ...
2552 2023-10-27 00:00:00+00:00       32532.189453
2553 2023-10-28 00:00:00+00:00       37668.542969
2554 2023-10-29 00:00:00+00:00       37703.910156
2555 2023-10-30 00:00:00+00:00       37437.492188
2556 2023-10-31 00:00:00+00:00       37472.789062

[2557 rows x 2 columns]


# System 9068

In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.386400,
	"longitude": -104.551200,
	"start_date": "2017-08-23",
	"end_date": "2025-04-30",
	"daily": "sunshine_duration",
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunshine_duration = daily.Variables(0).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

df_daily  = daily_dataframe.rename(columns = {'date':'time'})
df_daily['time'] = df_daily['time'].dt.date
df_daily
df_daily.to_csv("prize_weather/9068.csv", index = False)

Coordinates: 40.38664245605469°N -104.57748413085938°E
Elevation: 1405.0 m asl
Timezone: b'America/Denver'b'GMT-6'
Timezone difference to GMT+0: -21600s

Daily data
                           date  sunshine_duration
0    2017-08-23 00:00:00+00:00       41930.851562
1    2017-08-24 00:00:00+00:00       42661.921875
2    2017-08-25 00:00:00+00:00       46573.371094
3    2017-08-26 00:00:00+00:00       44795.289062
4    2017-08-27 00:00:00+00:00       45172.296875
...                        ...                ...
2803 2025-04-26 00:00:00+00:00       45950.511719
2804 2025-04-27 00:00:00+00:00       46789.804688
2805 2025-04-28 00:00:00+00:00       39789.828125
2806 2025-04-29 00:00:00+00:00       48274.183594
2807 2025-04-30 00:00:00+00:00       43023.144531

[2808 rows x 2 columns]
